In [1]:
%cd /Users/akarsh/Documents/projects/fastInfer/StoryForge

/Users/akarsh/Documents/projects/fastInfer/StoryForge


In [ ]:
import copy
import torch

from tokenizers import Tokenizer

from storyforge.config import load_config
from storyforge.training import load_initial_model

In [3]:
config = load_config("configs/default.toml")

In [4]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

In [5]:
masked_config = copy.deepcopy(config)
masked_config["model"]["story_boundary_masking"] = True

In [6]:
masked_model = load_initial_model(masked_config, device)
masked_model.eval()

ForgeGPT(
  (token): Embedding(8192, 384)
  (position): Embedding(256, 384)
  (drop): Dropout(p=0.1, inplace=False)
  (blocks): ModuleList(
    (0-5): 6 x TransformerBlock(
      (norm1): LayerNorm((384,), eps=1e-05, elementwise_affine=True, bias=True)
      (attn): CausalAttention(
        (qkv): Linear(in_features=384, out_features=1152, bias=True)
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (attn_drop): Dropout(p=0.1, inplace=False)
        (out_drop): Dropout(p=0.1, inplace=False)
      )
      (norm2): LayerNorm((384,), eps=1e-05, elementwise_affine=True, bias=True)
      (mlp): Sequential(
        (0): Linear(in_features=384, out_features=1536, bias=True)
        (1): GELU(approximate='none')
        (2): Linear(in_features=1536, out_features=384, bias=True)
        (3): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (norm): LayerNorm((384,), eps=1e-05, elementwise_affine=True, bias=True)
)

In [7]:
tokenizer = Tokenizer.from_file(
    config["tokenizer"]["path"]
)


In [8]:
eos_id = tokenizer.token_to_id(
    config["tokenizer"]["eos_token"]
)

In [ ]:
first_story = tokenizer.encode("Ben lost his hat.").ids
second_story = tokenizer.encode("Mia baked a cake.").ids

In [10]:
ids = torch.tensor(
    [first_story + [eos_id] + second_story + [eos_id]],
    dtype=torch.long,
    device=device,
)

In [11]:
second_start = len(first_story) + 1

In [ ]:
changed_ids = ids.clone()

In [ ]:
changed_ids[:, :len(first_story)] = tokenizer.token_to_id("!")

In [14]:
with torch.no_grad():
    original_logits, _ = masked_model(ids)
    changed_logits, _ = masked_model(changed_ids)

In [15]:
assert torch.allclose(
    original_logits[:, second_start:, :],
    changed_logits[:, second_start:, :],
    atol=1e-6,
    rtol=1e-5,
)

In [16]:
from storyforge.config import load_config

In [17]:
baseline_config = load_config("configs/default.toml")
masked_config = load_config("configs/story_masked.toml")

assert not baseline_config["model"].get(
    "story_boundary_masking",
    False,
)

assert masked_config["model"]["story_boundary_masking"] is True

assert (
    masked_config["checkpoints"]["initial_path"]
    == baseline_config["checkpoints"]["initial_path"]
)
